# LAK-8 — MERGE / upsert: Copy-on-Write vs Merge-on-Read

**Break → Detect → Fix → Prove.** A tiny upsert — one CDC row, one late correction — into a
**partitioned** table. Under **copy-on-write** (Iceberg's default) that 1-row `MERGE` **rewrites
every data file in the touched partition**. Under **merge-on-read** the same MERGE writes a small
**delete file** instead — cheap writes, but reads must reconcile deletes and you must compact.

**Pre-requisite:** the unified Spark server is up (`make up`) — Spark UI at http://localhost:4040.
This notebook connects via Spark Connect.

**Laptop-safe:** a few hundred rows across a handful of partitions, all under `.tmp/`; the notebook
drops both tables at the end (`make clean` clears the rest). This is a **write-amplification**
pathology, not a memory one, so the default **tuned** box is fine.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md) (LAK-8 row).

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

COW = "iceberg_catalog.default.lak8_cow"   # copy-on-write (Iceberg default)
MOR = "iceberg_catalog.default.lak8_mor"   # merge-on-read (delete files)
spark

## Step 0 — a small, partitioned orders table

We synthesize a few hundred `orders` and **partition by `bucket(8, customer_id)`** so a given
`customer_id` (and thus a 1-row correction for it) lands in exactly one partition. A small upsert
should touch **one** partition — that's what makes the CoW whole-partition rewrite visible.

In [ ]:
N_ROWS  = 400
N_BUCKETS = 8

def orders(lo, hi, status="new", seed=1):
    return (spark.range(lo, hi).withColumnRenamed("id", "order_id")
            .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(40)))
            .withColumn("amount", F.round(F.rand(seed=seed) * 100, 2))
            .withColumn("status", F.lit(status)))

# the single-row upsert both tables will receive (a "late correction" to order #7)
one_update = spark.createDataFrame([(7, 999.99, "corrected")],
                                   ["order_id", "amount", "status"])
one_update.createOrReplaceTempView("one_update")
one_update.show()

## 1. Break it — copy-on-write MERGE

Create `lak8_cow` **without** any MoR properties: copy-on-write is Iceberg's default write mode.
Load the rows, then capture its health *before* the upsert.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {COW}")
(orders(1, N_ROWS + 1).writeTo(COW).using("iceberg")
    .partitionedBy(F.bucket(N_BUCKETS, "customer_id"))
    .create())                                   # CoW by default — no write.*.mode props

cow_before = table_health(spark, COW, "CoW before MERGE")
print(f"loaded {spark.table(COW).count()} rows into {N_BUCKETS} buckets")
print("CoW before:", cow_before)

In [ ]:
# The 1-row MERGE. Under CoW this rewrites the whole data file(s) of the matched partition.
def cow_merge():
    spark.sql(
        f"MERGE INTO {COW} t USING one_update u ON t.order_id = u.order_id "
        f"WHEN MATCHED THEN UPDATE SET t.amount = u.amount, t.status = u.status"
    )

m_cow = measure(spark, "CoW 1-row MERGE", cow_merge)
cow_after = table_health(spark, COW, "CoW after MERGE")
print("amount for order_id=7 now:",
      spark.sql(f"SELECT amount FROM {COW} WHERE order_id = 7").first()["amount"])
print("CoW after:", cow_after)

## 2. Detect it — read the snapshot + `.files` metadata

This is a **metadata** tell, not a Spark UI Stages tell. The MERGE's own snapshot summary shows it
**added** new data files and **deleted** the old ones — a full rewrite of the touched partition,
far more than the one row that changed.

In [ ]:
def merge_snapshot(table):
    return spark.sql(
        f"SELECT operation, "
        f"summary['added-data-files']   AS added_data, "
        f"summary['deleted-data-files'] AS deleted_data, "
        f"summary['added-delete-files'] AS added_deletes, "
        f"summary['added-records']      AS added_records, "
        f"summary['added-files-size']   AS added_bytes "
        f"FROM {table}.snapshots ORDER BY committed_at DESC LIMIT 1"
    )

print("CoW — what the 1-row MERGE actually wrote (note added/deleted DATA files):")
merge_snapshot(COW).show(truncate=False)

print("CoW .files by content (0=data, 1=position-delete, 2=equality-delete):")
spark.sql(f"SELECT content, COUNT(*) AS files, SUM(file_size_in_bytes) AS bytes "
          f"FROM {COW}.files GROUP BY content ORDER BY content").show()

## 3. Fix / tradeoff — merge-on-read MERGE

Now the same table, created with **merge-on-read** write modes (format-version 2 is required for
delete files). The identical 1-row MERGE will write a tiny **delete file** instead of rewriting the
partition.

> Set the modes at **CREATE** via `tableProperty(...)`, or flip an existing table with
> `ALTER TABLE ... SET TBLPROPERTIES ('format-version'='2','write.merge.mode'='merge-on-read', ...)`.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {MOR}")
(orders(1, N_ROWS + 1).writeTo(MOR).using("iceberg")
    .partitionedBy(F.bucket(N_BUCKETS, "customer_id"))
    .tableProperty("format-version", "2")
    .tableProperty("write.merge.mode",  "merge-on-read")
    .tableProperty("write.update.mode", "merge-on-read")
    .tableProperty("write.delete.mode", "merge-on-read")
    .create())

mor_before = table_health(spark, MOR, "MoR before MERGE")
print("MoR before:", mor_before)

In [ ]:
def mor_merge():
    spark.sql(
        f"MERGE INTO {MOR} t USING one_update u ON t.order_id = u.order_id "
        f"WHEN MATCHED THEN UPDATE SET t.amount = u.amount, t.status = u.status"
    )

m_mor = measure(spark, "MoR 1-row MERGE", mor_merge)
mor_after = table_health(spark, MOR, "MoR after MERGE")

print("MoR — same MERGE now writes a DELETE file, not a rewritten data file:")
merge_snapshot(MOR).show(truncate=False)
print("MoR .files by content (look for content=1, a position-delete file):")
spark.sql(f"SELECT content, COUNT(*) AS files, SUM(file_size_in_bytes) AS bytes "
          f"FROM {MOR}.files GROUP BY content ORDER BY content").show()

## 4. Prove it

Same 1-row change, two write modes. CoW **rewrites a whole data file** (data files added *and*
deleted, real bytes written); MoR adds a **tiny delete file** and rewrites no data. The table
below makes the write-amplification gap concrete.

In [ ]:
print("Table health — CoW churns DATA files; MoR adds a DELETE file (see Data files / Total size):")
compare_health([cow_before, cow_after, mor_before, mor_after])

print("\nWrite cost of the identical 1-row MERGE (CoW rewrites a partition; MoR writes a delete vector):")
compare([m_cow, m_mor])

## 5. The MoR catch — reads pay, so compact

MoR's cheap write isn't free: every **read** must now reconcile the delete file against the data
file, and delete files **accumulate** with each upsert. Periodic compaction applies the deletes and
removes them — the same `rewrite_data_files` from **LAK-2**, here clearing **delete** files. After
it, `.files` collapses back to data-only (`content` = 0).

In [ ]:
spark.sql(
    "CALL iceberg_catalog.system.rewrite_data_files("
    "table => 'default.lak8_mor', "
    "options => map('delete-file-threshold','1'))"
).show(truncate=False)

print("MoR .files after compaction — delete files applied & removed (content should be 0 only):")
spark.sql(f"SELECT content, COUNT(*) AS files FROM {MOR}.files GROUP BY content ORDER BY content").show()
compare_health([mor_after, table_health(spark, MOR, "MoR after compaction")])

## Takeaways & "in real production…"

- **CoW = read-optimized, write-amplified.** A 1-row CoW update rewrites whole files — never apply
  CDC row-by-row to a CoW table.
- **MoR = write-optimized, needs compaction.** Cheap upserts, but reads reconcile deletes and delete
  files pile up — **schedule `rewrite_data_files`** to apply and clear them (ties to **LAK-2**).
- **Choose by read/write ratio.** Read-heavy + rare writes → CoW. Write-heavy / frequent small
  upserts → MoR + compaction. Same call dbt makes for incremental models (**DBT-2**).
- **Batch your upserts** to amortize the cost (one MERGE for many changes under CoW; compact on a
  cadence under MoR), and **partition so changes localize** to one partition, not the whole table.

## Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {COW}")
spark.sql(f"DROP TABLE IF EXISTS {MOR}")
print("Dropped lak8_cow + lak8_mor. (`make clean` clears all of .tmp.)")